# Análise Exploratória — Oportunidades em meio ao Caos

Esta etapa avalia a qualidade dos dados depois do pré-processamento e investiga padrões que podem ajudar na futura classificação de regimes de estresse e na análise de resiliência dos setores.

A pergunta principal do projeto continua sendo:

> **Dado um cenário de estresse socioeconômico, quais setores historicamente apresentaram maior resiliência e melhor relação entre retorno e risco?**

A EDA não tem como objetivo provar causalidade nem treinar o modelo final. O foco é entender a base, suas limitações e quais hipóteses fazem sentido levar para a próxima etapa.

## 1. Carregamento e configuração

O notebook procura primeiro os arquivos em `data/curated`. Se não encontrar, tenta ler o bucket `curated` do MinIO. Assim, o notebook funciona tanto localmente quanto com o armazenamento do projeto.

In [ ]:
from pathlib import Path
from io import BytesIO
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Descobre a raiz do projeto quando o notebook roda dentro de notebooks/.
CWD = Path.cwd().resolve()
if (CWD / "data" / "curated").exists():
    BASE = CWD
elif (CWD.parent / "data" / "curated").exists():
    BASE = CWD.parent
else:
    BASE = CWD

CURATED = BASE / "data" / "curated"
load_dotenv(BASE / ".env")

print("Diretório do projeto:", BASE)
print("Curated local:", CURATED)

In [ ]:
def read_curated(filename: str) -> pd.DataFrame:
    local = CURATED / filename
    if local.exists():
        print(f"LOCAL  -> {local}")
        return pd.read_csv(local)

    # Fallback opcional para MinIO.
    import boto3

    endpoint = os.getenv("MINIO_ENDPOINT", "localhost:9000")
    secure = os.getenv("MINIO_SECURE", "false").lower() in {"1", "true", "yes"}
    access = os.getenv("MINIO_ROOT_USER") or os.getenv("MINIO_ACCESS_KEY")
    secret = os.getenv("MINIO_ROOT_PASSWORD") or os.getenv("MINIO_SECRET_KEY")
    bucket = os.getenv("MINIO_BUCKET_CURATED", "curated")

    if not access or not secret:
        raise FileNotFoundError(
            f"{filename} não encontrado localmente e credenciais do MinIO não estão configuradas."
        )

    s3 = boto3.client(
        "s3",
        endpoint_url=f"http{'s' if secure else ''}://{endpoint}",
        aws_access_key_id=access,
        aws_secret_access_key=secret,
        region_name="us-east-1",
    )
    obj = s3.get_object(Bucket=bucket, Key=filename)
    print(f"MINIO  -> s3://{bucket}/{filename}")
    return pd.read_csv(BytesIO(obj["Body"].read()))

ref = read_curated("dataset_monthly_reference.csv")
mvp = read_curated("dataset_mvp.csv")
complete = read_curated("dataset_mvp_complete.csv")

for df in (ref, mvp, complete):
    df["date"] = pd.to_datetime(df["date"])
    df.sort_values("date", inplace=True)
    df.reset_index(drop=True, inplace=True)

print("\nShapes:")
print("reference:", ref.shape)
print("mvp:      ", mvp.shape)
print("complete: ", complete.shape)

## 2. Entendimento dos três datasets

- **`dataset_monthly_reference.csv`**: dados alinhados ao mês de referência. É o principal dataset para gráficos históricos e EDA.
- **`dataset_mvp.csv`**: contém os lags aplicados no pré-processamento para reduzir *look-ahead bias*. É a referência para avaliar futuras features do modelo.
- **`dataset_mvp_complete.csv`**: mantém somente meses em que todas as variáveis principais estão preenchidas. É útil para algumas análises conjuntas, mas não deve limitar toda a EDA.

In [ ]:
def dataset_summary(name, df):
    return {
        "dataset": name,
        "linhas": len(df),
        "colunas": df.shape[1],
        "inicio": df["date"].min(),
        "fim": df["date"].max(),
        "duplicadas_data": int(df["date"].duplicated().sum()),
        "missing_total": int(df.isna().sum().sum()),
    }

summary_datasets = pd.DataFrame([
    dataset_summary("monthly_reference", ref),
    dataset_summary("mvp", mvp),
    dataset_summary("mvp_complete", complete),
])
display(summary_datasets)

### Atenção ao último mês

O pré-processamento deve trabalhar apenas com meses completos. Como proteção adicional, a EDA identifica o último mês calendário já encerrado e evita usar um mês ainda em andamento nas comparações temporais.

In [ ]:
today = pd.Timestamp.today().normalize()
last_closed_month = today.to_period("M").start_time - pd.Timedelta(days=1)
last_closed_month = last_closed_month.to_period("M").to_timestamp("M")

print("Hoje:", today.date())
print("Último mês calendário encerrado:", last_closed_month.date())

for name, df in [("reference", ref), ("mvp", mvp), ("complete", complete)]:
    future_rows = df[df["date"] > last_closed_month]
    print(f"{name}: {len(future_rows)} linha(s) após o último mês encerrado")

# Bases de análise sem mês calendário incompleto.
ref_eda = ref[ref["date"] <= last_closed_month].copy()
mvp_eda = mvp[mvp["date"] <= last_closed_month].copy()
complete_eda = complete[complete["date"] <= last_closed_month].copy()

## 3. Qualidade dos dados

Aqui verificamos tipos, missing values, duplicidades e continuidade mensal. Missing não será preenchido automaticamente: primeiro precisamos entender por que existe.

In [ ]:
quality = pd.DataFrame({
    "coluna": ref_eda.columns,
    "tipo": [str(ref_eda[c].dtype) for c in ref_eda.columns],
    "non_null": [int(ref_eda[c].notna().sum()) for c in ref_eda.columns],
    "missing": [int(ref_eda[c].isna().sum()) for c in ref_eda.columns],
    "missing_pct": [round(ref_eda[c].isna().mean() * 100, 2) for c in ref_eda.columns],
})
display(quality.sort_values("missing_pct", ascending=False))

print("Linhas duplicadas completas:", ref_eda.duplicated().sum())
print("Datas duplicadas:", ref_eda["date"].duplicated().sum())

In [ ]:
# Verifica meses ausentes entre início e fim da base.
periods = ref_eda["date"].dt.to_period("M")
expected = pd.period_range(periods.min(), periods.max(), freq="M")
missing_months = expected.difference(pd.Index(periods.unique()))

print("Meses esperados:", len(expected))
print("Meses presentes:", periods.nunique())
print("Meses ausentes:", len(missing_months))
if len(missing_months):
    display(pd.DataFrame({"mes_ausente": missing_months.astype(str)}))

## 4. Cobertura histórica por variável

Essa análise ajuda a não perder histórico desnecessariamente. Uma variável pode ter poucos anos de cobertura sem impedir análises que não dependem dela.

In [ ]:
def coverage_table(df):
    rows = []
    for col in df.columns:
        if col == "date":
            continue
        valid = df.loc[df[col].notna(), ["date", col]]
        rows.append({
            "variavel": col,
            "inicio": valid["date"].min() if len(valid) else pd.NaT,
            "fim": valid["date"].max() if len(valid) else pd.NaT,
            "observacoes": int(df[col].notna().sum()),
            "missing": int(df[col].isna().sum()),
            "cobertura_pct": round(df[col].notna().mean() * 100, 2),
        })
    return pd.DataFrame(rows).sort_values(["cobertura_pct", "variavel"])

coverage = coverage_table(ref_eda)
display(coverage)

In [ ]:
# Compara a janela macroeconômica com a janela em que todos os setores estão disponíveis.
macro_cols = [
    "ibc_br", "selic", "usd_brl", "ipca_month", "ipca_12m", "pib_index", "unemployment"
]
sector_cols = ["ibovespa", "ifnc", "icon", "iee"]

macro_available = mvp_eda.dropna(subset=[c for c in macro_cols if c in mvp_eda.columns])
sector_available = ref_eda.dropna(subset=[c for c in sector_cols if c in ref_eda.columns])

comparison_history = pd.DataFrame([
    {
        "conjunto": "Macroeconômico completo",
        "linhas": len(macro_available),
        "inicio": macro_available["date"].min(),
        "fim": macro_available["date"].max(),
    },
    {
        "conjunto": "Todos os índices B3",
        "linhas": len(sector_available),
        "inicio": sector_available["date"].min(),
        "fim": sector_available["date"].max(),
    },
    {
        "conjunto": "MVP complete",
        "linhas": len(complete_eda),
        "inicio": complete_eda["date"].min(),
        "fim": complete_eda["date"].max(),
    },
])
display(comparison_history)

## 5. Estrutura das variáveis

As variáveis são separadas em dois papéis:

- **Regime econômico**: candidatas futuras para classificar períodos de normalidade ou estresse.
- **Mercado/resiliência**: usadas para medir como Ibovespa, IFNC, ICON e IEE se comportaram.

In [ ]:
regime_features = [c for c in [
    "ibc_br", "ibc_br_change", "selic", "selic_change",
    "usd_brl", "usd_brl_return", "usd_brl_volatility",
    "ipca_month", "ipca_12m", "pib_index", "pib_change_3m",
    "unemployment", "unemployment_change"
] if c in mvp_eda.columns]

market_features = [c for c in ref_eda.columns if any(
    c.startswith(prefix) for prefix in ["ibovespa", "ifnc", "icon", "iee"]
)]

print("Features de regime:")
print(regime_features)
print("\nVariáveis de mercado/resiliência:")
print(market_features)

## 6. Estatística descritiva

Além de média e mediana, usamos desvio padrão, quartis, assimetria e curtose. Assimetria ajuda a identificar distribuições inclinadas; curtose elevada pode indicar caudas pesadas e maior presença de valores extremos.

In [ ]:
main_numeric = [c for c in [
    "ibc_br_change", "selic", "selic_change", "usd_brl_return", "usd_brl_volatility",
    "ipca_12m", "pib_change_3m", "unemployment", "unemployment_change",
    "ibovespa_return_1m", "ifnc_return_1m", "icon_return_1m", "iee_return_1m"
] if c in mvp_eda.columns]

stats = pd.DataFrame(index=main_numeric)
stats["n"] = mvp_eda[main_numeric].count()
stats["media"] = mvp_eda[main_numeric].mean()
stats["mediana"] = mvp_eda[main_numeric].median()
stats["desvio_padrao"] = mvp_eda[main_numeric].std()
stats["min"] = mvp_eda[main_numeric].min()
stats["q25"] = mvp_eda[main_numeric].quantile(.25)
stats["q75"] = mvp_eda[main_numeric].quantile(.75)
stats["max"] = mvp_eda[main_numeric].max()
stats["assimetria"] = mvp_eda[main_numeric].skew()
stats["curtose"] = mvp_eda[main_numeric].kurt()
display(stats)

## 7. Distribuição das principais variáveis

Histogramas mostram concentração e formato da distribuição. Boxplots ajudam a localizar dispersão e valores extremos. Os extremos não são removidos: eles podem representar justamente momentos de estresse.

In [ ]:
dist_cols = [c for c in [
    "ibc_br_change", "selic", "usd_brl_return", "usd_brl_volatility",
    "ipca_12m", "pib_change_3m", "unemployment",
    "ifnc_return_1m", "icon_return_1m", "iee_return_1m"
] if c in mvp_eda.columns]

for col in dist_cols:
    s = mvp_eda[col].dropna()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(s, bins=min(20, max(8, int(np.sqrt(len(s))))), edgecolor="black", alpha=.75)
    ax.axvline(s.mean(), linestyle="--", label=f"Média {s.mean():.3f}")
    ax.axvline(s.median(), linestyle=":", label=f"Mediana {s.median():.3f}")
    ax.set_title(f"Distribuição — {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequência")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
return_cols = [c for c in ["ibovespa_return_1m", "ifnc_return_1m", "icon_return_1m", "iee_return_1m"] if c in ref_eda.columns]
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([ref_eda[c].dropna() for c in return_cols], tick_labels=return_cols, showfliers=True)
ax.axhline(0, linewidth=1)
ax.set_title("Distribuição dos retornos mensais — índices B3")
ax.set_ylabel("Retorno mensal")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 8. Evolução temporal

Gráficos de linha ajudam a observar tendência, mudanças de nível e possíveis quebras. Variáveis com escalas muito diferentes são mostradas separadamente.

In [ ]:
time_series_cols = [c for c in [
    "ibc_br", "selic", "usd_brl", "usd_brl_volatility", "ipca_12m", "pib_index", "unemployment"
] if c in ref_eda.columns]

for col in time_series_cols:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(ref_eda["date"], ref_eda[col])
    ax.set_title(f"Evolução temporal — {col}")
    ax.set_xlabel("Data")
    ax.set_ylabel(col)
    ax.grid(alpha=.25)
    plt.tight_layout()
    plt.show()

### Índices normalizados para base 100

Os índices possuem níveis diferentes. Para comparar apenas a trajetória relativa, cada série é normalizada para começar em 100. Os valores originais não são alterados.

In [ ]:
indices = [c for c in ["ibovespa", "ifnc", "icon", "iee"] if c in ref_eda.columns]
base_idx = ref_eda[["date"] + indices].dropna(subset=indices).copy()

normalized = base_idx[["date"]].copy()
for col in indices:
    normalized[col] = base_idx[col] / base_idx[col].iloc[0] * 100

fig, ax = plt.subplots(figsize=(12, 5))
for col in indices:
    ax.plot(normalized["date"], normalized[col], label=col.upper())
ax.set_title("Índices B3 normalizados — base 100")
ax.set_xlabel("Data")
ax.set_ylabel("Base 100")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## 9. Retorno, risco e comportamento dos setores

Nesta etapa aprofundamos a análise dos índices de mercado. O objetivo não é identificar apenas qual setor apresentou maior retorno histórico, mas avaliar conjuntamente retorno, volatilidade e frequência de perdas.

Essa distinção é importante para o conceito de resiliência adotado neste projeto: um setor potencialmente resiliente não deve ser avaliado somente pela sua capacidade de gerar retorno, mas também pela magnitude e frequência das perdas.

In [ ]:
# Métricas gerais de retorno e risco por índice.
sector_metrics = []
for index_name in ["ibovespa", "ifnc", "icon", "iee"]:
    ret_col = f"{index_name}_return_1m"
    vol_col = f"{index_name}_volatility_3m_ann"

    if ret_col not in ref_eda.columns:
        continue

    tmp_cols = ["date", ret_col]
    if vol_col in ref_eda.columns:
        tmp_cols.append(vol_col)

    tmp = ref_eda[tmp_cols].dropna(subset=[ret_col]).copy()

    sector_metrics.append({
        "indice": index_name.upper(),
        "observacoes": len(tmp),
        "retorno_medio": tmp[ret_col].mean(),
        "retorno_mediano": tmp[ret_col].median(),
        "volatilidade_retorno": tmp[ret_col].std(),
        "melhor_mes": tmp[ret_col].max(),
        "pior_mes": tmp[ret_col].min(),
        "meses_positivos_pct": (tmp[ret_col] > 0).mean(),
        "meses_negativos_pct": (tmp[ret_col] < 0).mean(),
        "volatilidade_media_3m": (
            tmp[vol_col].mean() if vol_col in tmp.columns else np.nan
        ),
    })

sector_metrics = pd.DataFrame(sector_metrics)

display(
    sector_metrics.style.format({
        "retorno_medio": "{:.2%}",
        "retorno_mediano": "{:.2%}",
        "volatilidade_retorno": "{:.2%}",
        "melhor_mes": "{:.2%}",
        "pior_mes": "{:.2%}",
        "meses_positivos_pct": "{:.1%}",
        "meses_negativos_pct": "{:.1%}",
        "volatilidade_media_3m": "{:.2%}",
    })
)

## 10. Drawdown e velocidade de recuperação

O drawdown mede a distância entre o nível atual do índice e seu maior valor histórico observado até aquele momento.

Para a análise de resiliência, entretanto, a profundidade da perda não é suficiente. Também é relevante avaliar quanto tempo o índice leva para recuperar o nível anterior ao drawdown.

Nesta etapa analisamos:

- drawdown máximo;
- data do drawdown máximo;
- nível anterior à perda;
- tempo necessário para recuperação;
- percentual de períodos em drawdown.

A hipótese exploratória é que setores mais resilientes apresentem perdas menores e/ou recuperação mais rápida após períodos adversos.

In [ ]:
def recovery_metrics(df, index_name):
    value_col = index_name

    if value_col not in df.columns:
        return None

    tmp = df[["date", value_col]].dropna().copy()
    tmp = tmp.sort_values("date").reset_index(drop=True)

    # Máximo histórico até cada ponto.
    tmp["peak"] = tmp[value_col].cummax()

    # Drawdown.
    tmp["drawdown_calc"] = tmp[value_col] / tmp["peak"] - 1

    # Pior drawdown.
    worst_idx = tmp["drawdown_calc"].idxmin()
    worst_row = tmp.loc[worst_idx]

    peak_before = worst_row["peak"]
    peak_date = tmp.loc[
        tmp.loc[:worst_idx, value_col].idxmax(), "date"
    ]

    # Procura o primeiro momento posterior em que o índice
    # retorna ao nível do pico anterior.
    recovery = tmp.loc[
        (tmp.index > worst_idx) &
        (tmp[value_col] >= peak_before)
    ]

    if recovery.empty:
        recovery_date = pd.NaT
        recovery_months = np.nan
    else:
        recovery_date = recovery.iloc[0]["date"]
        recovery_months = (
            (recovery_date.year - peak_date.year) * 12
            + recovery_date.month - peak_date.month
        )

    return {
        "indice": index_name.upper(),
        "pior_drawdown": worst_row["drawdown_calc"],
        "data_pior_drawdown": worst_row["date"],
        "data_pico_anterior": peak_date,
        "data_recuperacao": recovery_date,
        "meses_para_recuperar": recovery_months,
        "meses_em_drawdown": (tmp["drawdown_calc"] < 0).mean(),
    }


recovery_rows = []

for index_name in ["ibovespa", "ifnc", "icon", "iee"]:
    result = recovery_metrics(ref_eda, index_name)
    if result:
        recovery_rows.append(result)

recovery_table = pd.DataFrame(recovery_rows)

display(
    recovery_table.style.format({
        "pior_drawdown": "{:.2%}",
        "meses_em_drawdown": "{:.1%}",
    })
)


fig, ax = plt.subplots(figsize=(11, 5))

for index_name in ["ibovespa", "ifnc", "icon", "iee"]:
    dd_col = f"{index_name}_drawdown"

    if dd_col in ref_eda.columns:
        ax.plot(
            ref_eda["date"],
            ref_eda[dd_col],
            label=index_name.upper()
        )

ax.axhline(0, linewidth=1)
ax.set_title("Drawdown dos índices B3")
ax.set_xlabel("Data")
ax.set_ylabel("Drawdown")
ax.legend()
ax.grid(alpha=.25)

plt.tight_layout()
plt.show()

## 11. Identificação exploratória de períodos de stress

Antes da construção do modelo de classificação, é importante investigar se os próprios dados apresentam períodos caracterizados simultaneamente por deterioração econômica, aumento de inflação, pressão cambial, aumento do desemprego e/ou maior volatilidade.

Nesta etapa será construído um indicador exploratório denominado **Stress Score**.

O indicador não representa ainda o target definitivo do modelo. Seu objetivo é:

1. identificar períodos potencialmente estressados;
2. comparar esses períodos com eventos econômicos conhecidos;
3. gerar hipóteses para a definição posterior do target `stress`.

As variáveis serão transformadas em scores padronizados para permitir a comparação entre indicadores com escalas diferentes.

In [ ]:
# Variáveis e direção esperada de estresse.
# +1 significa que valores altos representam maior estresse.
# -1 significa que valores baixos representam maior estresse.

stress_components = {
    "ibc_br_change": -1,
    "selic_change": 1,
    "usd_brl_return": 1,
    "usd_brl_volatility": 1,
    "ipca_12m": 1,
    "unemployment_change": 1,
    "pib_change_3m": -1,
}

stress_data = mvp_eda[["date"]].copy()

for col, direction in stress_components.items():
    if col not in mvp_eda.columns:
        continue

    series = mvp_eda[col]

    # Padronização.
    z = (series - series.mean()) / series.std()

    # Ajusta direção econômica do indicador.
    stress_data[f"{col}_stress_z"] = z * direction

stress_score_cols = [
    c for c in stress_data.columns
    if c.endswith("_stress_z")
]

# Média dos componentes disponíveis.
stress_data["stress_score"] = stress_data[stress_score_cols].mean(axis=1)

display(
    stress_data[
        ["date", "stress_score"] + stress_score_cols
    ].sort_values("stress_score", ascending=False).head(20)
)

In [ ]:
# Classificação exploratória baseada nos percentis do Stress Score.
# Não utilizar esta classificação como target definitivo ainda.

q75 = stress_data["stress_score"].quantile(.75)
q90 = stress_data["stress_score"].quantile(.90)

stress_data["stress_level"] = np.select(
    [
        stress_data["stress_score"] >= q90,
        stress_data["stress_score"] >= q75,
    ],
    [
        "Stress elevado",
        "Atenção",
    ],
    default="Normal"
)

display(
    stress_data[
        ["date", "stress_score", "stress_level"]
    ].sort_values("stress_score", ascending=False).head(30)
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    stress_data["date"],
    stress_data["stress_score"],
    label="Stress Score"
)

ax.axhline(q75, linestyle="--", label="Percentil 75")
ax.axhline(q90, linestyle=":", label="Percentil 90")

ax.set_title("Stress Score exploratório ao longo do tempo")
ax.set_xlabel("Data")
ax.set_ylabel("Stress Score padronizado")
ax.legend()
ax.grid(alpha=.25)

plt.tight_layout()
plt.show()

## 12. Validação exploratória dos períodos de stress

O Stress Score permite identificar períodos sem depender exclusivamente de uma classificação manual.

Nesta etapa comparamos:

- períodos históricos previamente definidos;
- meses classificados pelo Stress Score;
- intensidade média do estresse em cada período.

A finalidade é verificar se o indicador apresenta coerência econômica e se os períodos historicamente reconhecidos como adversos aparecem entre os valores mais elevados.

Essa comparação é exploratória e não representa validação causal.

In [ ]:
periods = {
    "Recessão 2015-2016": ("2015-01-01", "2016-12-31"),
    "Pré-pandemia 2018-2019": ("2018-01-01", "2019-12-31"),
    "Pandemia 2020": ("2020-01-01", "2020-12-31"),
    "Inflação/juros 2021-2022": ("2021-01-01", "2022-12-31"),
    "2023 em diante": ("2023-01-01", str(last_closed_month.date())),
}

def period_metrics(df, period_name, start, end, index_name):
    ret_col = f"{index_name}_return_1m"
    dd_col = f"{index_name}_drawdown"
    tmp = df.loc[df["date"].between(pd.Timestamp(start), pd.Timestamp(end)), ["date", ret_col, dd_col]].dropna(subset=[ret_col])
    if tmp.empty:
        return None
    compounded = (1 + tmp[ret_col]).prod() - 1
    return {
        "periodo": period_name,
        "indice": index_name.upper(),
        "meses": len(tmp),
        "retorno_acumulado": compounded,
        "retorno_medio_mensal": tmp[ret_col].mean(),
        "volatilidade_mensal": tmp[ret_col].std(),
        "pior_drawdown_observado": tmp[dd_col].min(),
        "melhor_mes": tmp[ret_col].max(),
        "pior_mes": tmp[ret_col].min(),
    }

rows = []
for pname, (start, end) in periods.items():
    for idx_name in ["ibovespa", "ifnc", "icon", "iee"]:
        r = period_metrics(ref_eda, pname, start, end, idx_name)
        if r:
            rows.append(r)
period_table = pd.DataFrame(rows)
display(period_table)

In [ ]:
# Reutiliza os períodos históricos já definidos no notebook.
stress_period_rows = []

for period_name, (start, end) in periods.items():
    tmp = stress_data[
        stress_data["date"].between(
            pd.Timestamp(start),
            pd.Timestamp(end)
        )
    ].copy()

    if tmp.empty:
        continue

    stress_period_rows.append({
        "periodo": period_name,
        "meses": len(tmp),
        "stress_medio": tmp["stress_score"].mean(),
        "stress_maximo": tmp["stress_score"].max(),
        "pct_stress_elevado": (
            tmp["stress_level"].eq("Stress elevado").mean()
        ),
        "pct_atencao_ou_stress": (
            tmp["stress_level"].isin(
                ["Atenção", "Stress elevado"]
            ).mean()
        ),
    })

stress_period_table = pd.DataFrame(stress_period_rows)

display(
    stress_period_table.style.format({
        "stress_medio": "{:.2f}",
        "stress_maximo": "{:.2f}",
        "pct_stress_elevado": "{:.1%}",
        "pct_atencao_ou_stress": "{:.1%}",
    })
)

In [ ]:
# Meses com maior Stress Score.
top_stress_months = (
    stress_data[
        ["date", "stress_score", "stress_level"]
    ]
    .sort_values("stress_score", ascending=False)
    .head(15)
)

display(top_stress_months)

## 13. Stress × Normal: comportamento dos setores

A principal pergunta desta análise é:

> **Os setores apresentam comportamentos diferentes quando o ambiente macroeconômico está sob maior estresse?**

Para responder exploratoriamente, os meses serão divididos em:

- **Normal**: Stress Score abaixo do percentil 75;
- **Stress**: Stress Score igual ou acima do percentil 75.

Para cada índice serão comparados:

- retorno médio;
- retorno acumulado;
- volatilidade;
- frequência de meses negativos;
- pior retorno mensal.

A classificação será utilizada apenas para análise exploratória. O target definitivo do modelo será definido posteriormente.

In [ ]:
# Junta o regime exploratório com os dados de mercado.
analysis_regime = ref_eda.merge(
    stress_data[["date", "stress_score", "stress_level"]],
    on="date",
    how="left"
)

analysis_regime["regime_eda"] = np.where(
    analysis_regime["stress_score"] >= q75,
    "Stress",
    "Normal"
)

stress_normal_rows = []

for index_name in ["ibovespa", "ifnc", "icon", "iee"]:
    ret_col = f"{index_name}_return_1m"

    if ret_col not in analysis_regime.columns:
        continue

    for regime in ["Normal", "Stress"]:
        tmp = analysis_regime[
            analysis_regime["regime_eda"] == regime
        ][["date", ret_col]].dropna()

        if tmp.empty:
            continue

        stress_normal_rows.append({
            "indice": index_name.upper(),
            "regime": regime,
            "meses": len(tmp),
            "retorno_medio": tmp[ret_col].mean(),
            "retorno_acumulado": (1 + tmp[ret_col]).prod() - 1,
            "volatilidade": tmp[ret_col].std(),
            "meses_negativos_pct": (tmp[ret_col] < 0).mean(),
            "pior_mes": tmp[ret_col].min(),
        })

stress_normal = pd.DataFrame(stress_normal_rows)

display(
    stress_normal.style.format({
        "retorno_medio": "{:.2%}",
        "retorno_acumulado": "{:.2%}",
        "volatilidade": "{:.2%}",
        "meses_negativos_pct": "{:.1%}",
        "pior_mes": "{:.2%}",
    })
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

pivot = stress_normal.pivot(
    index="indice",
    columns="regime",
    values="retorno_medio"
)

pivot.plot(
    kind="bar",
    ax=ax
)

ax.axhline(0, linewidth=1)
ax.set_title("Retorno médio mensal — Normal × Stress")
ax.set_xlabel("Índice")
ax.set_ylabel("Retorno médio")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 14. Análise de defasagens entre indicadores macroeconômicos e setores

Nesta etapa investigamos se alterações nos indicadores macroeconômicos apresentam associação mais forte com os retornos setoriais no mesmo mês ou após algum período de defasagem.

São avaliadas defasagens de 0, 1, 3 e 6 meses.

A análise possui caráter exploratório e busca identificar possíveis relações temporais que possam orientar a construção das features do modelo de Machine Learning.

Importante: correlação defasada não implica causalidade. Os resultados serão utilizados para formular hipóteses e selecionar variáveis para análises posteriores.


In [ ]:
# ============================================================
# 14. ANÁLISE DE DEFASAGENS MACROECONÔMICAS
# ============================================================

macro_lag_features = [
    "ibc_br_change",
    "selic_change",
    "usd_brl_return",
    "usd_brl_volatility",
    "ipca_12m",
    "unemployment_change",
    "pib_change_3m",
]

sector_lag_features = [
    "ibovespa_return",
    "ifnc_return",
    "icon_return",
    "iee_return",
]

available_macro = [
    c for c in macro_lag_features
    if c in ref_eda.columns
]

available_sectors = [
    c for c in sector_lag_features
    if c in ref_eda.columns
]

lags = [0, 1, 3, 6]

lag_results = []

for macro in available_macro:

    for sector in available_sectors:

        for lag in lags:

            temp = ref_eda[[macro, sector]].copy()

            if lag > 0:
                temp[macro] = temp[macro].shift(lag)

            temp = temp.dropna()

            if len(temp) < 10:
                continue

            pearson = temp[macro].corr(
                temp[sector],
                method="pearson"
            )

            spearman = temp[macro].corr(
                temp[sector],
                method="spearman"
            )

            lag_results.append({
                "macro": macro,
                "setor": sector,
                "lag_meses": lag,
                "observacoes": len(temp),
                "pearson": pearson,
                "spearman": spearman,
                "abs_pearson": abs(pearson),
                "abs_spearman": abs(spearman)
            })

lag_results = pd.DataFrame(lag_results)

display(
    lag_results
    .sort_values(
        ["setor", "abs_pearson"],
        ascending=[True, False]
    )
    .head(30)
)


### 14.1 Melhor defasagem por relação

Para cada combinação entre indicador macroeconômico e setor, selecionamos a defasagem que apresentou a maior correlação absoluta.

O objetivo não é afirmar que essa é uma relação causal, mas identificar possíveis janelas temporais relevantes para a modelagem.


In [ ]:
best_lags = (
    lag_results
    .sort_values(
        ["macro", "setor", "abs_pearson"],
        ascending=[True, True, False]
    )
    .drop_duplicates(
        subset=["macro", "setor"]
    )
    .sort_values(
        "abs_pearson",
        ascending=False
    )
)

display(best_lags)


### Interpretação

As relações encontradas nesta etapa devem ser tratadas como hipóteses para o modelo.

Uma correlação mais forte em determinada defasagem pode indicar que o mercado incorpora mudanças macroeconômicas de maneira gradual, ou simplesmente refletir tendências comuns entre as séries.

Essas relações deverão ser posteriormente avaliadas respeitando a disponibilidade temporal das informações, evitando look-ahead bias.


## 15. Correlação entre indicadores macroeconômicos e setores por regime

A correlação entre variáveis pode mudar significativamente durante períodos de estresse.

Por isso, nesta etapa comparamos as relações macroeconômicas e setoriais em três perspectivas:

- Todos os períodos;
- Períodos classificados como Normal;
- Períodos classificados como Stress.

O objetivo é verificar se determinados setores apresentam comportamento diferente justamente nos momentos de maior pressão econômica.


In [ ]:
# ============================================================
# 15. CORRELAÇÃO POR REGIME
# ============================================================

analysis_regime = ref_eda.copy()

analysis_regime = analysis_regime.merge(
    stress_data[["date", "stress_score"]],
    on="date",
    how="left"
)

q75 = analysis_regime["stress_score"].quantile(0.75)

analysis_regime["regime_eda"] = np.where(
    analysis_regime["stress_score"] >= q75,
    "Stress",
    "Normal"
)

regime_correlations = []

for macro in available_macro:

    for sector in available_sectors:

        for regime in ["Todos", "Normal", "Stress"]:

            if regime == "Todos":
                temp = analysis_regime[[macro, sector]].dropna()
            else:
                temp = analysis_regime[
                    analysis_regime["regime_eda"] == regime
                ][[macro, sector]].dropna()

            if len(temp) < 10:
                continue

            regime_correlations.append({
                "macro": macro,
                "setor": sector,
                "regime": regime,
                "observacoes": len(temp),
                "pearson": temp[macro].corr(temp[sector]),
                "spearman": temp[macro].corr(
                    temp[sector],
                    method="spearman"
                )
            })

regime_correlations = pd.DataFrame(
    regime_correlations
)

display(regime_correlations)


### 15.1 Exemplo: relação entre Selic e setores

A seguir avaliamos visualmente se a relação entre variações da Selic e retornos setoriais se modifica entre os regimes Normal e Stress.


In [ ]:
# ============================================================
# 15.1 EXEMPLO VISUAL — SELIC
# ============================================================

if "selic_change" in available_macro:

    pivot_selic = (
        regime_correlations[
            regime_correlations["macro"] == "selic_change"
        ]
        .pivot(
            index="setor",
            columns="regime",
            values="pearson"
        )
    )

    display(pivot_selic)

    pivot_selic[
        ["Normal", "Stress"]
    ].plot(
        kind="bar",
        figsize=(12, 6)
    )

    plt.axhline(0, linewidth=1)
    plt.title(
        "Correlação entre Variação da Selic e Retorno Setorial"
    )
    plt.ylabel("Correlação de Pearson")
    plt.xlabel("Setor")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


### Interpretação

A comparação entre regimes permite verificar se as relações observadas no período completo permanecem estáveis durante momentos de estresse.

Mudanças relevantes entre Normal e Stress podem indicar que determinados setores possuem maior sensibilidade às condições macroeconômicas justamente nos períodos mais críticos.


## 16. Outliers e eventos econômicos

Outliers financeiros não devem ser automaticamente removidos.

Em um estudo sobre resiliência, movimentos extremos podem representar justamente os eventos que queremos compreender.

Nesta etapa identificamos meses com retornos setoriais excepcionalmente positivos ou negativos utilizando o critério do intervalo interquartil (IQR).

Posteriormente relacionamos esses eventos ao Stress Score exploratório para verificar se movimentos extremos ocorreram com maior frequência durante períodos de estresse.


In [ ]:
# ============================================================
# 16. OUTLIERS COMO EVENTOS ECONÔMICOS
# ============================================================

outlier_events = []

for sector in available_sectors:

    q1 = ref_eda[sector].quantile(0.25)
    q3 = ref_eda[sector].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (
        (ref_eda[sector] < lower_bound) |
        (ref_eda[sector] > upper_bound)
    )

    temp = ref_eda.loc[
        mask,
        ["date", sector]
    ].copy()

    temp["setor"] = sector
    temp["tipo"] = np.where(
        temp[sector] < lower_bound,
        "Perda extrema",
        "Ganho extremo"
    )

    outlier_events.append(
        temp.rename(
            columns={sector: "retorno"}
        )
    )

outlier_events = pd.concat(
    outlier_events,
    ignore_index=True
)

outlier_events = outlier_events.merge(
    stress_data[["date", "stress_score"]],
    on="date",
    how="left"
)

outlier_events = outlier_events.merge(
    analysis_regime[["date", "regime_eda"]],
    on="date",
    how="left"
)

outlier_events = outlier_events.sort_values(
    "date"
)

display(outlier_events)


### 16.1 Resumo dos eventos extremos

Nesta tabela verificamos quantos eventos extremos ocorreram em cada setor e qual era o nível de estresse socioeconômico observado nesses meses.


In [ ]:
# ============================================================
# 16.1 RESUMO DOS EVENTOS EXTREMOS
# ============================================================

outlier_summary = (
    outlier_events
    .groupby("setor")
    .agg(
        eventos=("retorno", "count"),
        stress_medio=("stress_score", "mean"),
        stress_maximo=("stress_score", "max"),
        percentual_stress=(
            "regime_eda",
            lambda x: (
                x.eq("Stress").mean() * 100
            )
        )
    )
    .reset_index()
)

display(
    outlier_summary.sort_values(
        "percentual_stress",
        ascending=False
    )
)


### Interpretação

Os outliers representam potenciais eventos de mercado relevantes para a análise de resiliência.

Uma concentração de perdas extremas em períodos classificados como Stress pode reforçar a relação entre condições macroeconômicas adversas e comportamento setorial.

Por outro lado, setores que apresentam menor frequência de perdas extremas durante Stress podem constituir candidatos a maior resiliência.


## 17. Sazonalidade e comparação entre subperíodos

Nesta etapa investigamos dois aspectos complementares:

1. Se existe algum padrão sistemático de comportamento dos setores ao longo dos meses do ano;
2. Se o comportamento de retorno e risco permanece semelhante em diferentes subperíodos históricos.

A sazonalidade possui caráter complementar na análise, enquanto a comparação entre subperíodos permite verificar a estabilidade dos resultados ao longo do tempo.


In [ ]:
# ============================================================
# 17. SAZONALIDADE
# ============================================================

seasonal_data = ref_eda.copy()

seasonal_data["mes"] = (
    seasonal_data["date"].dt.month
)

seasonal_summary = (
    seasonal_data
    .groupby("mes")[available_sectors]
    .mean()
)

display(seasonal_summary)


In [ ]:
# ============================================================
# 17.1 GRÁFICO DE SAZONALIDADE
# ============================================================

seasonal_summary.plot(
    figsize=(14, 7),
    marker="o"
)

plt.title(
    "Retorno Médio dos Setores por Mês do Ano"
)
plt.xlabel("Mês")
plt.ylabel("Retorno médio")
plt.xticks(range(1, 13))
plt.axhline(0, linewidth=1)
plt.legend(
    title="Setor",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()


### 17.2 Comparação entre subperíodos

A comparação por subperíodos permite verificar se as conclusões observadas para todo o histórico são consistentes ou se são fortemente influenciadas por determinados eventos.

Os períodos históricos definidos anteriormente são utilizados aqui como segmentos descritivos.


In [ ]:
# ============================================================
# 17.2 COMPARAÇÃO ENTRE SUBPERÍODOS
# ============================================================

display(subperiod_summary)


In [ ]:
# ============================================================
# 17.3 RISCO POR SUBPERÍODO
# ============================================================

risk_summary = []

for period_name, period_info in periods.items():

    start = pd.Timestamp(period_info["start"])
    end = pd.Timestamp(period_info["end"])

    temp = ref_eda[
        (ref_eda["date"] >= start) &
        (ref_eda["date"] <= end)
    ]

    for sector in available_sectors:

        risk_summary.append({
            "periodo": period_name,
            "setor": sector,
            "observacoes": temp[sector].count(),
            "retorno_medio": temp[sector].mean(),
            "volatilidade": temp[sector].std(),
            "percentual_negativo": (
                (temp[sector] < 0).mean() * 100
            ),
            "pior_mes": temp[sector].min()
        })

risk_summary = pd.DataFrame(
    risk_summary
)

display(risk_summary)


### Interpretação

A análise de subperíodos permite avaliar a estabilidade dos resultados.

Caso um setor apresente bom desempenho apenas em um período específico, sua aparente resiliência pode estar associada a características particulares daquele contexto.

Já um comportamento consistente em diferentes períodos de estresse constitui evidência exploratória mais forte para a hipótese de resiliência.


## 18. Índice exploratório de resiliência setorial

Nesta etapa consolidamos os principais indicadores observados durante os períodos classificados como Stress em um índice exploratório de resiliência.

O objetivo é criar uma visão comparativa entre os setores considerando simultaneamente:

- retorno médio durante Stress;
- volatilidade;
- frequência de meses negativos;
- pior retorno mensal;
- drawdown;
- velocidade de recuperação.


In [ ]:
# ============================================================
# 18. ÍNDICE EXPLORATÓRIO DE RESILIÊNCIA
# ============================================================

stress_only = analysis_regime[
    analysis_regime["regime_eda"] == "Stress"
].copy()

resilience_base = []

for sector in available_sectors:

    if sector not in stress_only.columns:
        continue

    series = stress_only[sector].dropna()

    if len(series) == 0:
        continue

    resilience_base.append({
        "setor": sector,
        "observacoes_stress": len(series),
        "retorno_medio_stress": series.mean(),
        "volatilidade_stress": series.std(),
        "percentual_negativo_stress": (
            (series < 0).mean() * 100
        ),
        "pior_mes_stress": series.min()
    })

resilience_base = pd.DataFrame(
    resilience_base
)

display(resilience_base)


### 18.1 Drawdown observado durante Stress

Além do retorno e da volatilidade, incorporamos o drawdown como medida de perda acumulada.

Quanto menor a magnitude do drawdown, maior a evidência exploratória de preservação de capital durante períodos adversos.


In [ ]:
# ============================================================
# 18.1 DRAWDOWN DURANTE STRESS
# ============================================================

drawdown_stress = []

for sector in available_sectors:

    temp = analysis_regime[
        analysis_regime["regime_eda"] == "Stress"
    ][["date", sector]].dropna()

    if len(temp) == 0:
        continue

    cumulative = (
        (1 + temp[sector])
        .cumprod()
    )

    peak = cumulative.cummax()

    drawdown = (
        cumulative / peak - 1
    )

    drawdown_stress.append({
        "setor": sector,
        "drawdown_stress": drawdown.min()
    })

drawdown_stress = pd.DataFrame(
    drawdown_stress
)

resilience_base = resilience_base.merge(
    drawdown_stress,
    on="setor",
    how="left"
)

display(resilience_base)


### 18.2 Inclusão da velocidade de recuperação

A capacidade de recuperar perdas também é relevante para a definição de resiliência.

Setores que sofrem uma perda, mas retornam rapidamente aos níveis anteriores podem apresentar características diferentes daqueles que permanecem deprimidos por vários meses.


In [ ]:
# ============================================================
# 18.2 MERGE COM TEMPO DE RECUPERAÇÃO
# ============================================================

if "recovery_table" in globals():

    recovery_for_resilience = (
        recovery_table[
            [
                "setor",
                "meses_para_recuperar"
            ]
        ]
        .rename(
            columns={
                "meses_para_recuperar":
                    "tempo_recuperacao"
            }
        )
    )

    resilience_base = resilience_base.merge(
        recovery_for_resilience,
        on="setor",
        how="left"
    )

else:

    resilience_base["tempo_recuperacao"] = np.nan

display(resilience_base)


### 18.3 Normalização dos indicadores

Os indicadores possuem escalas diferentes e, por isso, são transformados para uma escala comparável entre 0 e 1.

Para todos os componentes buscamos interpretar:

- maior retorno = melhor;
- menor volatilidade = melhor;
- menor frequência de perdas = melhor;
- menor perda extrema = melhor;
- menor drawdown = melhor;
- menor tempo de recuperação = melhor.


In [ ]:
# ============================================================
# 18.3 NORMALIZAÇÃO
# ============================================================

def minmax_score(series, higher_is_better=True):

    series = series.astype(float)

    min_value = series.min()
    max_value = series.max()

    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(
            np.nan,
            index=series.index
        )

    if max_value == min_value:
        return pd.Series(
            1.0,
            index=series.index
        )

    score = (
        (series - min_value) /
        (max_value - min_value)
    )

    if not higher_is_better:
        score = 1 - score

    return score


resilience_base["score_retorno"] = (
    minmax_score(
        resilience_base["retorno_medio_stress"],
        higher_is_better=True
    )
)

resilience_base["score_volatilidade"] = (
    minmax_score(
        resilience_base["volatilidade_stress"],
        higher_is_better=False
    )
)

resilience_base["score_perdas"] = (
    minmax_score(
        resilience_base["percentual_negativo_stress"],
        higher_is_better=False
    )
)

resilience_base["score_pior_mes"] = (
    minmax_score(
        resilience_base["pior_mes_stress"],
        higher_is_better=True
    )
)

resilience_base["score_drawdown"] = (
    minmax_score(
        resilience_base["drawdown_stress"],
        higher_is_better=True
    )
)

resilience_base["score_recuperacao"] = (
    minmax_score(
        resilience_base["tempo_recuperacao"],
        higher_is_better=False
    )
)


### 18.4 Cálculo do Índice

O Índice final é calculado pela média dos componentes disponíveis.

O resultado é posteriormente transformado para uma escala de 0 a 100, facilitando a interpretação e comparação entre setores.

Quanto maior o valor, maior a resiliência observada de forma exploratória durante os períodos classificados como Stress.


In [ ]:
# ============================================================
# 18.4 ÍNDICE FINAL
# ============================================================

score_columns = [
    "score_retorno",
    "score_volatilidade",
    "score_perdas",
    "score_pior_mes",
    "score_drawdown",
    "score_recuperacao"
]

resilience_base["resilience_score"] = (
    resilience_base[score_columns]
    .mean(axis=1, skipna=True)
)

resilience_base["resilience_score_100"] = (
    resilience_base["resilience_score"] * 100
)

resilience_ranking = (
    resilience_base
    .sort_values(
        "resilience_score_100",
        ascending=False
    )
    .reset_index(drop=True)
)

resilience_ranking["ranking"] = (
    resilience_ranking.index + 1
)

display(
    resilience_ranking[
        [
            "ranking",
            "setor",
            "observacoes_stress",
            "retorno_medio_stress",
            "volatilidade_stress",
            "percentual_negativo_stress",
            "pior_mes_stress",
            "drawdown_stress",
            "tempo_recuperacao",
            "resilience_score_100"
        ]
    ]
)


In [ ]:
# ============================================================
# 18.5 VISUALIZAÇÃO DO RANKING
# ============================================================

plot_data = (
    resilience_ranking
    .sort_values(
        "resilience_score_100",
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

plt.barh(
    plot_data["setor"],
    plot_data["resilience_score_100"]
)

plt.xlabel(
    "Índice exploratório de resiliência (0–100)"
)

plt.ylabel("Setor")

plt.title(
    "Ranking Exploratório de Resiliência durante Períodos de Stress"
)

plt.tight_layout()
plt.show()


### 18.6 Interpretação e conexão com o problema de pesquisa

O ranking sintetiza as principais evidências obtidas durante a EDA.

Setores com maior pontuação apresentam, dentro dos critérios adotados, uma combinação mais favorável de retorno, risco, perdas e recuperação durante períodos classificados como Stress.

Entretanto, o Índice deve ser interpretado como uma ferramenta exploratória e não como uma conclusão definitiva de resiliência.

Os resultados serão utilizados para orientar as próximas etapas do projeto, especialmente:

- definição das variáveis utilizadas no modelo;
- construção de features defasadas;
- definição e comparação de diferentes estratégias para classificação de períodos de Stress;
- avaliação da capacidade preditiva dos modelos;
- comparação entre setores em diferentes cenários econômicos.

Importante: o `resilience_score_100` não deve ser utilizado como target do modelo de Machine Learning. Ele representa uma síntese exploratória construída a partir dos próprios indicadores analisados na EDA.


## 19. Volume disponível para Machine Learning

O número de linhas não é a única questão. Também importa quantas features serão usadas, quanto histórico cada uma possui e, futuramente, quantos meses serão rotulados como `stress` e `normal`.

Nesta etapa ainda não existe o target `stress`, então não é possível avaliar o balanceamento das classes.

In [ ]:
initial_feature_set = [c for c in [
    "ibc_br_change", "selic", "selic_change", "usd_brl_return", "usd_brl_volatility",
    "ipca_12m", "pib_change_3m", "unemployment", "unemployment_change"
] if c in mvp_eda.columns]

regime_ready = mvp_eda[["date"] + initial_feature_set].dropna()
all_market_ready = ref_eda[["date"] + sector_cols].dropna()

volume = pd.DataFrame([
    {
        "dataset": "MVP completo (todas core)",
        "linhas": len(complete_eda),
        "features/series": 11,
        "inicio": complete_eda["date"].min(),
        "fim": complete_eda["date"].max(),
    },
    {
        "dataset": "Features iniciais de regime",
        "linhas": len(regime_ready),
        "features/series": len(initial_feature_set),
        "inicio": regime_ready["date"].min(),
        "fim": regime_ready["date"].max(),
    },
    {
        "dataset": "Todos os índices B3",
        "linhas": len(all_market_ready),
        "features/series": len(sector_cols),
        "inicio": all_market_ready["date"].min(),
        "fim": all_market_ready["date"].max(),
    },
])
display(volume)

print("Features candidatas ao primeiro modelo:")
print(initial_feature_set)
print("\nObservações completas nessas features:", len(regime_ready))
print("Target 'stress' existe no dataset?", "stress" in mvp_eda.columns)

## 20. Dataset de regime x dataset de resiliência

A EDA permite avaliar uma separação importante:

- **Dataset de regime**: features macroeconômicas e socioeconômicas para classificar `stress/normal`.
- **Dataset de resiliência**: índices B3 e métricas de retorno, volatilidade e drawdown para comparar os setores.

Essa divisão evita perder histórico macroeconômico só porque um índice setorial possui menos dados e também deixa mais claro o papel de cada variável.

In [ ]:
regime_cols = ["date"] + initial_feature_set
resilience_cols = ["date"] + [c for c in ref_eda.columns if any(c.startswith(p) for p in ["ibovespa", "ifnc", "icon", "iee"])]

regime_dataset_preview = mvp_eda[regime_cols].dropna().copy()
resilience_dataset_preview = ref_eda[resilience_cols].dropna(subset=sector_cols).copy()

print("Dataset de regime (preview):", regime_dataset_preview.shape)
print(regime_dataset_preview["date"].min(), "->", regime_dataset_preview["date"].max())
print("Dataset de resiliência (preview):", resilience_dataset_preview.shape)
print(resilience_dataset_preview["date"].min(), "->", resilience_dataset_preview["date"].max())

## 21. Resumo automático dos principais achados

O bloco abaixo não substitui a interpretação. Ele resume alguns fatos objetivos para facilitar a documentação da entrega.

In [ ]:
# Cobertura mais limitante entre as principais séries.
core_for_coverage = [c for c in macro_cols + sector_cols if c in coverage["variavel"].values]
core_cov = coverage[coverage["variavel"].isin(core_for_coverage)].sort_values("observacoes")
limiting = core_cov.iloc[0]

# Setor com menor drawdown observado (mais negativo = pior perda).
dd_df = pd.DataFrame(worst_dd)
worst_sector = dd_df.loc[dd_df["pior_drawdown"].idxmin()]
best_sector = dd_df.loc[dd_df["pior_drawdown"].idxmax()]

print(f"Período da EDA: {ref_eda['date'].min().date()} a {ref_eda['date'].max().date()}")
print(f"Variável core com menor cobertura: {limiting['variavel']} ({int(limiting['observacoes'])} observações)")
print(f"Observações completas nas features iniciais de regime: {len(regime_ready)}")
print(f"Observações com todos os índices B3: {len(all_market_ready)}")
print(f"Maior queda observada entre os drawdowns: {worst_sector['indice']} = {worst_sector['pior_drawdown']:.2%}")
print(f"Menor pior-drawdown entre os índices comparados: {best_sector['indice']} = {best_sector['pior_drawdown']:.2%}")

## 22. Hipóteses para a próxima etapa

As hipóteses abaixo devem ser confirmadas ou descartadas usando os resultados apresentados nas células anteriores:

1. A volatilidade cambial pode aumentar em períodos em que os índices também apresentam maior risco.
2. IFNC, ICON e IEE podem responder de formas diferentes a mudanças na Selic.
3. ICON pode apresentar maior sensibilidade a deterioração de atividade econômica e mercado de trabalho.
4. A combinação de inflação alta, juros altos, câmbio e desaceleração pode ser mais útil para identificar regimes do que uma única variável isolada.
5. Retorno, volatilidade e drawdown devem ser analisados em conjunto antes de classificar um setor como resiliente.

Estas são hipóteses exploratórias, não conclusões causais.

## 23. Limitações identificadas

Pontos que devem ser considerados na próxima fase:

- frequência mensal reduz o número de observações;
- observações de séries temporais não são totalmente independentes;
- diferentes séries possuem coberturas históricas diferentes;
- PIB é trimestral e precisa de tratamento temporal;
- lags usados no MVP são aproximações conservadoras das datas reais de divulgação;
- são analisados apenas três índices setoriais na primeira versão;
- o target `stress` ainda precisa ser definido e documentado;
- modelos complexos podem sofrer overfitting com esse volume de dados.

A futura validação deve respeitar a ordem temporal, evitando um `train_test_split` aleatório comum.

## 24. Próximos passos

Depois desta EDA, a sequência sugerida é:

1. revisar os insights e anomalias encontradas;
2. definir formalmente o critério de `stress = 0/1`;
3. verificar a quantidade de meses em cada classe;
4. fechar o conjunto inicial de features;
5. criar um baseline;
6. testar Logistic Regression;
7. comparar com Random Forest controlando a complexidade;
8. usar validação temporal;
9. aplicar os regimes identificados à análise de IFNC, ICON e IEE;
10. construir uma medida/ranking de resiliência.